# Customer Churn Prediction — Telecom Services

Predicting which telecom customers are likely to cancel their subscription (churn), so the business can step in early with a attractive offers.

## Section 1: Problem & Context

**Why churn matters.** Nowadays most of telecom companies are subscription based and making money so losing a customer isn't just one lost sale it’s about losing their all future payments It's also well known in the industry that it acquire a lot of cost and difficult to get a new customer than an existing customer. So if we can identify customers who are *about to* leave so the retention team can reach out to them with a discounts, offers something more on their current plan or just a support call before they cancel.

**How ML helps.** Instead of waiting for customers to churn we train a model on our previous customer data on people who already churned or stayed to find patterns like which contract types, tenure lengths, or add-on services are reasons why customers moving to other company. Then we use that model to calculate current customers and rank them by churn risk.

**About the data.** We're using the Telco Customer Churn dataset from Kaggle:

https://www.kaggle.com/datasets/blastchar/telco-customer-churn

This is a view of customer’s accounts from a fake telecom company. Each row is one customer with columns describing their demographics (gender, senior citizen status), account info (tenure, contract type, payment method) the services signed up for phone, internet, streaming, tech support, etc and their charges. The target column is Churn which is Yes if the customer left in the last month and No. if they're still a customer. For modelling we'll convert this to 1 (churned) and 0 means stayed.

**The ML task** This is a binary classification problem for each customer predict `Churn = 1` or `Churn = 0`, and ideally also get a probability score so we can rank customers by risk instead of just yes/no.

## Section 2: Data Exploration & Quality Analysis (EDA)

Before I do anything else I want to just look at the data. How many rows and columns, what type each column is, and if something looks broken like missing values.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Dataset from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
df = pd.read_csv("file.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# check datatypes and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

One thing I noticed is `TotalCharges` is showing up as text, not a number. That's weird because it should just be dollars. Let me check for missing values the normal way first, and then find out why this column is not numeric.

In [ ]:
# the normal check says 0 missing, but that's misleading, keep reading below
print(df.isnull().sum().sum(), "missing values found by isnull()")

# TotalCharges is text, so maybe some rows are just empty spaces instead of a real NaN
blank_mask = df["TotalCharges"].str.strip() == ""
print("rows where TotalCharges is just blank:", blank_mask.sum())
df[blank_mask][["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

So `isnull()` said 0 missing, but that's only because these 11 rows have a blank string `" "` instead of an actual NaN, so pandas doesn't count it as missing. All 11 of them have `tenure = 0`, which makes sense to me — they are brand new customers, first month, so there is no total charge yet. I will fix this properly in Section 3. For now I'm just noting it as a data quality problem.

Next let me check the target column itself.

In [ ]:
# how many customers churned vs stayed, in percent
churn_counts = df["Churn"].value_counts(normalize=True) * 100
print(churn_counts)

churn_counts.plot(kind="bar", color=["steelblue", "indianred"])
plt.title("Churn Rate")
plt.ylabel("% of customers")
plt.xticks(rotation=0)
plt.show()

Around 26% of customers churned and 74% stayed. This is not super extreme but it is an imbalance, so I need to be careful.

If I just predicted "No churn" for literally every customer, I would already get ~74% accuracy, but that model would be useless since it never actually finds the people who are about to leave. So accuracy alone is a bad metric here, it hides how the model does on the smaller class which is the one we actually care about.

Because of this I will look more at:
- **Recall** — out of all the customers who really churned, how many did the model catch. Missing a churner means we lose that customer for good.
- **Precision** — out of all the customers the model flagged as risky, how many actually churned. This matters because sending retention discounts costs money, so we don't want to give discounts to people who were never going to leave anyway.
- **ROC-AUC** — a general score of how well the model separates churners from non-churners.

I will come back to the precision vs recall trade-off again in Section 5.

In [ ]:
# quick look at tenure and contract type against churn
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=df, x="tenure", hue="Churn", multiple="stack", ax=axes[0])
axes[0].set_title("Tenure (months) vs Churn")

sns.countplot(data=df, x="Contract", hue="Churn", ax=axes[1])
axes[1].set_title("Contract Type vs Churn")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

Two things stand out to me here:

- **Tenure**: a lot of churn happens in the first few months. Once a customer stays for a year or two, they seem much less likely to leave.
- **Contract type**: month-to-month customers churn a lot more than people on 1-year or 2-year contracts. Makes sense, a monthly plan is easy to cancel, a 2-year contract kind of locks you in.

So `tenure` and `Contract` already look like important features. I'll keep this in mind when I check feature importance later.

## Section 3: Preprocessing & Feature Engineering

Now I need to actually fix the problems I found and get the data ready for a model. Plan: fix `TotalCharges`, drop the ID column since it is not useful for predicting anything, make 2 new features, turn categories into numbers, scale the numeric columns, then split into train and test.

In [ ]:
data = df.copy()

# turn the blank strings into NaN, then convert to number, then just fill with 0
# (these are the tenure=0 people, so 0 total charge makes sense anyway)
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"].str.strip(), errors="coerce")
data["TotalCharges"] = data["TotalCharges"].fillna(0)

# customerID is just a label, not useful for the model
data = data.drop(columns=["customerID"])

# turn target into 0/1 so the model can use it
data["Churn"] = (data["Churn"] == "Yes").astype(int)

print("missing values left:", data.isnull().sum().sum())
data[["tenure", "TotalCharges", "MonthlyCharges", "Churn"]].head()

### Making 2 new features

1. **`tenure_group`** — puts tenure (months) into yearly buckets. I already saw in Section 2 that new customers churn a lot more, so grouping tenure into years should make that pattern easier for the model to pick up, especially for Logistic Regression.
2. **`total_services`** — just counts how many of the extra services (OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies) a customer has turned on. My guess is customers with more add-ons feel more "locked in" and churn less.

In [ ]:
# 1. group tenure (months) into years
bins = [0, 12, 24, 36, 48, 60, 72]
labels = ["0-1yr", "1-2yr", "2-3yr", "3-4yr", "4-5yr", "5-6yr"]
data["tenure_group"] = pd.cut(data["tenure"], bins=bins, labels=labels, include_lowest=True)

# 2. count how many add-on services are turned on ("Yes")
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]
data["total_services"] = (data[addon_cols] == "Yes").sum(axis=1)

data[["tenure", "tenure_group", "total_services"]].head()

### Turning categories into numbers, scaling, and splitting

Models only understand numbers, so all the text columns (`Contract`, `PaymentMethod`, `InternetService`, my new `tenure_group`, etc.) need to become dummy 0/1 columns. The numeric columns (`tenure`, `MonthlyCharges`, `TotalCharges`, `total_services`) get scaled with `StandardScaler` so one big-range column doesn't dominate the others, mainly for Logistic Regression.

One thing I learned that's easy to mess up: I split into train/test **before** scaling, and I only fit the scaler on the training data. If I scaled everything first and then split, the test set would leak into the training process through the mean/std used for scaling.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# turn all text/category columns into dummy 0/1 columns
data_encoded = pd.get_dummies(data, drop_first=True)

X = data_encoded.drop(columns=["Churn"])
y = data_encoded["Churn"]

# split first, scale after, so test data doesn't leak into training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ["tenure", "MonthlyCharges", "TotalCharges", "total_services"]
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("train shape:", X_train.shape, " test shape:", X_test.shape)
print("train churn rate: {:.1%}  test churn rate: {:.1%}".format(y_train.mean(), y_test.mean()))